# Evaluación final en test

Primera y única vez que se toca el test set. Se comparan los 3 modelos
finales: NBSVM, BETO baseline y BETO + LoRA, cada uno con su mejor
configuración ya entrenada.

In [ ]:
!git clone https://github.com/camistrika/BETO_HUMOR.git
%cd BETO_HUMOR
!pip install -e . -q

In [ ]:
!pip install -q torchao --upgrade
!pip install -q transformers peft datasets scikit-learn pyyaml

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

from betohumor.config import DataConfig, BetoConfig, NbsvmConfig
from betohumor.utils import set_seed
from betohumor.dataset import load_and_split, HahaDataset
from betohumor.metrics import (
    nb_log_ratio,
    get_classification_report_df,
    get_confusion_matrix,
)
from betohumor.models.nbsvm import build_nbsvm_vectorizer, build_nbsvm_classifier
from betohumor.evaluate import predict
from betohumor.plots import plot_confusion_matrix, plot_grid_search_comparison

## 1. Datos (test set — recién ahora se usa)

In [ ]:
data_config = DataConfig()
beto_config = BetoConfig()
set_seed(data_config.seed)

df_train, df_val, df_test = load_and_split(data_config)
tokenizer = AutoTokenizer.from_pretrained(beto_config.base_model)

y_test = df_test[data_config.label_col].to_numpy()

## 2. NBSVM — reentrenar sobre train+val y predecir en test
Es barato, así que se reentrena directo en vez de guardar el modelo serializado.

In [ ]:
nbsvm_config = NbsvmConfig()

df_combined = pd.concat([df_train, df_val]).reset_index(drop=True)
text_col, label_col = data_config.text_col, data_config.label_col

X_train = df_combined[text_col].tolist()
y_train = df_combined[label_col].to_numpy()
X_test  = df_test[text_col].tolist()

vectorizer = build_nbsvm_vectorizer(nbsvm_config)
x_train = vectorizer.fit_transform(X_train)
x_test  = vectorizer.transform(X_test)

r = np.log(nb_log_ratio(x_train, 1, y_train) / nb_log_ratio(x_train, 0, y_train))

clf = build_nbsvm_classifier(nbsvm_config, seed=data_config.seed)
clf.fit(x_train.multiply(r), y_train)

preds_nbsvm = clf.predict(x_test.multiply(r))

## 3. BETO HEAD - cargar modelo final y predecir en test

In [ ]:
model_baseline = AutoModelForSequenceClassification.from_pretrained('results/final_beto')
test_dataset_baseline = HahaDataset(df_test, tokenizer, data_config)

preds_baseline, probs_baseline = predict(model_baseline, test_dataset_baseline)

## 4. BETO + LoRA - cargar modelo final y predecir en test

In [ ]:
base_model_for_lora = AutoModelForSequenceClassification.from_pretrained(beto_config.base_model, num_labels=beto_config.num_labels)
model_lora = PeftModel.from_pretrained(base_model_for_lora, 'results/final_lora')
test_dataset_lora = HahaDataset(df_test, tokenizer, data_config)

preds_lora, probs_lora = predict(model_lora, test_dataset_lora)

## 5. Tabla comparativa

In [ ]:
df_comparison = pd.DataFrame([
    {
        'modelo': 'NBSVM',
        'accuracy': accuracy_score(y_test, preds_nbsvm),
        'macro_f1': f1_score(y_test, preds_nbsvm, average='macro'),
    },
    {
        'modelo': 'BETO baseline',
        'accuracy': accuracy_score(y_test, preds_baseline),
        'macro_f1': f1_score(y_test, preds_baseline, average='macro'),
    },
    {
        'modelo': 'BETO + LoRA',
        'accuracy': accuracy_score(y_test, preds_lora),
        'macro_f1': f1_score(y_test, preds_lora, average='macro'),
    },
])

os.makedirs('results', exist_ok=True)
df_comparison.to_csv('results/final_comparison.csv', index=False)
df_comparison

## 6. Gráfico comparativo

In [ ]:
plot_grid_search_comparison(
    list(df_comparison['modelo']),
    list(df_comparison['macro_f1']),
    title='Comparación final — Macro F1 en test',
)

## 7. Classification report y matriz de confusión por modelo

In [ ]:
print('=== NBSVM ===')
display(get_classification_report_df(y_test, preds_nbsvm))
cm = get_confusion_matrix(y_test, preds_nbsvm, normalize='true')
plot_confusion_matrix(cm, title='Matriz de confusión — NBSVM (Test)')

In [ ]:
print('=== BETO baseline ===')
display(get_classification_report_df(y_test, preds_baseline))
cm = get_confusion_matrix(y_test, preds_baseline, normalize='true')
plot_confusion_matrix(cm, title='Matriz de confusión — BETO baseline (Test)')

In [ ]:
print('=== BETO + LoRA ===')
display(get_classification_report_df(y_test, preds_lora))
cm = get_confusion_matrix(y_test, preds_lora, normalize='true')
plot_confusion_matrix(cm, title='Matriz de confusión — BETO + LoRA (Test)')

## 8. Guardar predicciones


In [ ]:
df_preds = df_test.copy()
df_preds['pred_nbsvm']    = preds_nbsvm
df_preds['pred_baseline'] = preds_baseline
df_preds['prob_baseline'] = probs_baseline[:, 1]
df_preds['pred_lora']     = preds_lora
df_preds['prob_lora']     = probs_lora[:, 1]

df_preds.to_csv('results/test_predictions.csv', index=False)
print('Predicciones guardadas en results/test_predictions.csv')